# Decoder performance — baseline (2026-05-06)

Per-channel MAE, MAPE comparison (standard vs range-normalised), and waveform inspection for 3 test examples.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from dataset import CVDataset
from decoder import CVSurrogate

In [ ]:
CHECKPOINT  = ROOT / 'checkpoints/baseline/checkpoint_epoch10.pt'
STATS_PATH  = ROOT / 'norm_stats.json'
DATA_DIR    = Path('/media/8TBNVME/data/neh10/hdf5/cv8/simset_10M_cv8Eed_20260314/test')
MANIFEST    = DATA_DIR.parent / 'manifest_test.json'
N_SIMS      = 256
SIM_INDICES = [0, 1, 2]
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

WAVE_NAMES_CONT = [
    'Pap','Pas','Pla','Plv','Pra','Prv','Pvp','Pvs',
    'Qap','Qas','Qla','Qlv','Qra','Qrv','Qvp','Qvs',
    'Vap','Vas','Vla','Vlv','Vra','Vrv','Vvp','Vvs',
]
VALVE_NAMES = ['AV', 'MV', 'PV', 'TV']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Load model

In [ ]:
ckpt  = torch.load(CHECKPOINT, map_location=device)
model = CVSurrogate().to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f"Epoch {ckpt['epoch']}  val_loss={ckpt['val_loss']:.6f}")

with open(STATS_PATH) as f:
    stats = json.load(f)
with open(MANIFEST) as f:
    manifest = json.load(f)

## Run inference on test set

In [ ]:
ds     = CVDataset(DATA_DIR, manifest['index'][:N_SIMS], stats)
loader = DataLoader(ds, batch_size=256, num_workers=0)

all_pred, all_gt = [], []
with torch.no_grad():
    for params_b, waves_cont_b, _ in loader:
        pred_cont_b, _ = model(params_b.to(device))
        all_pred.append(pred_cont_b.cpu())
        all_gt.append(waves_cont_b)
ds.close()

all_pred = torch.cat(all_pred).numpy()   # (N, 24, 201)
all_gt   = torch.cat(all_gt).numpy()

wave_means = np.array([stats['waves'][n]['mean'] for n in WAVE_NAMES_CONT])
wave_stds  = np.array([stats['waves'][n]['std']  for n in WAVE_NAMES_CONT])
pred_phys  = all_pred * wave_stds[None,:,None] + wave_means[None,:,None]
gt_phys    = all_gt   * wave_stds[None,:,None] + wave_means[None,:,None]
print(f'Evaluated {N_SIMS} test simulations')

## Compute metrics

In [ ]:
abs_err = np.abs(pred_phys - gt_phys)          # (N, 24, 201)

sim_ch_mae = abs_err.mean(axis=2)              # (N, 24)
ch_mae     = sim_ch_mae.mean(axis=0)           # (24,)
ch_med_ae  = np.median(abs_err, axis=(0, 2))   # (24,)

pct_err     = abs_err / (np.abs(gt_phys) + 1e-6) * 100
ch_mape     = pct_err.mean(axis=(0, 2))        # (24,)
ch_med_mape = np.median(pct_err, axis=(0, 2))  # (24,)

sig_range   = np.maximum(gt_phys.max(axis=2) - gt_phys.min(axis=2), 1e-6)  # (N, 24)
rng_pct_err = abs_err / sig_range[:, :, None] * 100
ch_rne      = rng_pct_err.mean(axis=(0, 2))    # (24,)
ch_med_rne  = np.median(rng_pct_err, axis=(0, 2))  # (24,)

print(f"{'Channel':<8} {'Mean MAE':>10} {'Med MAE':>10} {'Mean MAPE':>11} {'Med MAPE':>10} {'Mean RNE':>10} {'Med RNE':>9}")
print('-' * 75)
for i, name in enumerate(WAVE_NAMES_CONT):
    print(f'{name:<8} {ch_mae[i]:>10.4f} {ch_med_ae[i]:>10.4f} {ch_mape[i]:>10.2f}% {ch_med_mape[i]:>9.2f}% {ch_rne[i]:>9.2f}% {ch_med_rne[i]:>8.2f}%')

## Error distributions

In [ ]:
x = np.arange(len(WAVE_NAMES_CONT))
w = 0.4
fig, axes = plt.subplots(2, 2, figsize=(20, 10))

axes[0, 0].bar(x - w/2, ch_mae,    width=w, color='steelblue', alpha=0.85, label='Mean MAE')
axes[0, 0].bar(x + w/2, ch_med_ae, width=w, color='darkcyan',  alpha=0.85, label='Median MAE')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(WAVE_NAMES_CONT, rotation=45, ha='right')
axes[0, 0].set_title('MAE — mean vs median per channel')
axes[0, 0].set_ylabel('AE (physical units)')
axes[0, 0].legend()

axes[0, 1].bar(x - w/2, ch_mape,     width=w, color='tomato',    alpha=0.85, label='Mean MAPE')
axes[0, 1].bar(x + w/2, ch_med_mape, width=w, color='darkorange', alpha=0.85, label='Median MAPE')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(WAVE_NAMES_CONT, rotation=45, ha='right')
axes[0, 1].set_title('MAPE — mean vs median  (inflated for zero-crossing Q/V)')
axes[0, 1].set_ylabel('MAPE (%)')
axes[0, 1].legend()

axes[1, 0].bar(x - w/2, ch_rne,     width=w, color='mediumseagreen', alpha=0.85, label='Mean RNE')
axes[1, 0].bar(x + w/2, ch_med_rne, width=w, color='darkgreen',      alpha=0.85, label='Median RNE')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(WAVE_NAMES_CONT, rotation=45, ha='right')
axes[1, 0].set_title('Range-normalised error — mean vs median  (safe for zero-crossing)')
axes[1, 0].set_ylabel('RNE (%)')
axes[1, 0].legend()

axes[1, 1].axis('off')

fig.suptitle(f"Baseline — error distributions  (epoch {ckpt['epoch']}, n={N_SIMS})", fontsize=13)
plt.tight_layout()
plt.show()

## Waveforms — 3 test examples

In [ ]:
ds3     = CVDataset(DATA_DIR, [manifest['index'][i] for i in SIM_INDICES], stats)
samples = [ds3[j] for j in range(len(SIM_INDICES))]
ds3.close()

pred_conts, pred_valves, gt_conts, gt_valves = [], [], [], []
for params, waves_cont, waves_valve in samples:
    with torch.no_grad():
        pc, pv = model(params.unsqueeze(0).to(device))
    pred_conts.append(pc.squeeze(0).cpu().numpy())
    pred_valves.append(pv.squeeze(0).cpu().numpy())
    gt_conts.append(waves_cont.numpy())
    gt_valves.append(waves_valve.numpy())

t = np.arange(201)

In [ ]:
ncols = len(SIM_INDICES)
nrows = len(WAVE_NAMES_CONT)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 2.5 * nrows), sharex=True)
for col, idx in enumerate(SIM_INDICES):
    for row, name in enumerate(WAVE_NAMES_CONT):
        ax = axes[row, col]
        ax.plot(t, gt_conts[col][row], color='steelblue', label='GT')
        ax.plot(t, pred_conts[col][row], color='tomato', linestyle='--', label='Pred')
        if col == 0:
            ax.set_ylabel(name, fontsize=8)
        if row == 0:
            ax.set_title(f'Test sim {idx}', fontsize=9)
        if row == nrows - 1:
            ax.set_xlabel('Time step', fontsize=8)
        if col == ncols - 1 and row == 0:
            ax.legend(loc='upper right', fontsize=7)
fig.suptitle(f"Baseline — continuous waveforms (epoch {ckpt['epoch']})", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(4, ncols, figsize=(6 * ncols, 8), sharex=True)
for col, idx in enumerate(SIM_INDICES):
    for row, name in enumerate(VALVE_NAMES):
        ax = axes[row, col]
        ax.step(t, gt_valves[col][row], color='steelblue', where='post', label='GT')
        ax.step(t, (pred_valves[col][row] > 0).astype(float), color='tomato', where='post', linestyle='--', label='Pred')
        ax.set_ylim(-0.1, 1.1)
        if col == 0:
            ax.set_ylabel(name, fontsize=8)
        if row == 0:
            ax.set_title(f'Test sim {idx}', fontsize=9)
        if row == 3:
            ax.set_xlabel('Time step', fontsize=8)
        if col == ncols - 1 and row == 0:
            ax.legend(loc='upper right', fontsize=7)
fig.suptitle(f"Baseline — valve signals (epoch {ckpt['epoch']})", fontsize=12)
plt.tight_layout()
plt.show()